# Fase 2b: Validación del Sistema de Recuperación Vectorial (Retrieval Semántico)

## 1. Introducción
Este notebook tiene como objetivo validar la base de datos vectorial recién creada (`noticias_tfg_vectores`). Implementaremos la lógica de búsqueda basada en la similitud del coseno (kNN - K-Nearest Neighbors).

## 2. Diferencia con BM25
A diferencia del Notebook 02 (BM25), aquí no buscaremos coincidencias exactas de palabras. 
1. Convertiremos la pregunta del usuario en un vector matemático de 1024 dimensiones usando `BAAI/bge-m3`.
2. Elasticsearch calculará la distancia espacial entre el vector de la pregunta y el medio millón de vectores de las noticias.
3. Devolverá las noticias más "cercanas" conceptualmente.

In [1]:
from elasticsearch import Elasticsearch

# Conexión a través del túnel SSH hacia valencia
ES_HOST = "http://localhost:9250"
INDEX_VECTORIAL = "noticias_tfg_vectores"

print(f"Conectando a Elasticsearch en {ES_HOST}...")
es = Elasticsearch(ES_HOST)

if es.ping():
    print(f"Conexión exitosa. Versión: {es.info()['version']['number']}")
    # Comprobamos que el índice está listo
    count = es.count(index=INDEX_VECTORIAL)['count']
    print(f"Documentos disponibles para búsqueda semántica: {count}")
else:
    print("Error: No se puede conectar. Revisa el túnel SSH.")

Conectando a Elasticsearch en http://localhost:9250...
Conexión exitosa. Versión: 8.12.0
Documentos disponibles para búsqueda semántica: 490956


In [2]:
import torch
from sentence_transformers import SentenceTransformer

print("Cargando modelo BGE-M3 (1024 dims) para procesar las preguntas...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Tiene que ser OBLIGATORIAMENTE el mismo modelo que usamos en la ingesta (Notebook 01b)
modelo_embeddings = SentenceTransformer('BAAI/bge-m3', device=device)
print(f"Modelo listo en: {device.upper()}")

/home/javierruiz/miniconda3/envs/environment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando modelo BGE-M3 (1024 dims) para procesar las preguntas...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3109.75it/s, Materializing param=pooler.dense.weight]                               


Modelo listo en: CUDA


In [3]:
def retrieve_context_vectorial(pregunta, top_k=5):
    """
    Convierte la pregunta en vector y busca las noticias más similares en Elasticsearch.
    """
    print(f"\nPregunta original: '{pregunta}'")
    print("Vectorizando pregunta y calculando distancias espaciales...")
    
    # 1. Transformamos la pregunta de texto a matemáticas (Vector de 1024 dimensiones)
    vector_pregunta = modelo_embeddings.encode(pregunta).tolist()
    
    # 2. Construimos la consulta kNN (K-Nearest Neighbors) para Elasticsearch
    query_knn = {
        "field": "vector_texto",
        "query_vector": vector_pregunta,
        "k": top_k,
        "num_candidates": 100 # Explora 100 candidatos cercanos y devuelve los 'k' mejores
    }
    
    # 3. Ejecutamos la búsqueda
    res = es.search(
        index=INDEX_VECTORIAL, 
        knn=query_knn,
        _source=["title", "source", "date", "url"], # Traemos metadatos
        size=top_k
    )
    
    hits = res['hits']['hits']
    
    # 4. Imprimimos los resultados formateados
    print(f"\nTop {top_k} resultados semánticos:")
    print("-" * 60)
    for i, hit in enumerate(hits, 1):
        score = hit['_score'] # En kNN, el score es la similitud coseno (1.0 = idéntico)
        source_data = hit['_source']
        
        print(f"[{i}] Score de Similitud: {score:.4f}")
        print(f"Título: {source_data.get('title', 'Sin título')}")
        print(f"Fuente: {source_data.get('source', 'Desconocida')}")
        print(f"Fecha:  {source_data.get('date', 'Sin fecha')}")
        print("-" * 60)
        
    return hits

In [6]:
# PRUEBA 1: Factual directa (Igual que en el BM25 para comparar)
q1 = "¿Qué porcentaje total de aranceles impuso Trump a China en abril de 2025?"
resultados_1 = retrieve_context_vectorial(q1, top_k=3)

# PRUEBA 2: Prueba Semántica / Sinónimos 
# (BM25 sufriría aquí porque no usamos las palabras clave del título)
q2 = "¿Cuánto dinero perdió Elon Musk tras el anuncio de los aranceles?"
resultados_2 = retrieve_context_vectorial(q2, top_k=3)

# PRUEBA 3: Inferencia compleja
q3 = "¿Qué medidas tomó Macron tras perder estrepitosamente en las votaciones europeas?"
resultados_3 = retrieve_context_vectorial(q3, top_k=3)


Pregunta original: '¿Qué porcentaje total de aranceles impuso Trump a China en abril de 2025?'
Vectorizando pregunta y calculando distancias espaciales...

Top 3 resultados semánticos:
------------------------------------------------------------
[1] Score de Similitud: 0.8731
Título: Resumen de noticias de aranceles de EE.UU. y Trump, 4 de abril 2025
Fuente: CNN en Español
Fecha:  2025-04-04 12:54:20.000000
------------------------------------------------------------
[2] Score de Similitud: 0.8591
Título: Trump amenaza con doblar los aranceles a China al 104% si no retira los suyos contra EE.UU.
Fuente: ABC.es
Fecha:  2025-04-07 17:45:35.000000
------------------------------------------------------------
[3] Score de Similitud: 0.8565
Título: La guerra de aranceles de Donald Trump - 8 de abril de 2025
Fuente: Ediciones EL PAÍS S.L.
Fecha:  2025-04-08 07:22:10.000000
------------------------------------------------------------

Pregunta original: '¿Cuánto dinero perdió Elon Musk tras e